# 34 · 生成评估：Faithfulness / 框架 / 基准

> 检索再好，回答满嘴跑火车照样完蛋。生成质量用**忠实度与相关性**衡量——最主流的方式是**让 LLM 当裁判**（LLM-as-a-Judge）。

**本文件覆盖知识点**：Faithfulness / Answer Relevance / Context Relevance / LLM-as-a-Judge / RAGAS / DeepEval / TruLens / LangSmith / 数据集 MS MARCO / BEIR

In [1]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


In [2]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

if not _HAS_KEY:
    print(NO_KEY_TIP)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集.md 是「人工标注的答案」，不能进索引 —— 否则第 33 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里显式排除。
_EXCLUDE = {'评测集.md'}

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name in _EXCLUDE:
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if not _CACHE_FILE.exists():
        return {}
    try:
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    except Exception as e:                      # 文件损坏（例如多进程同时写）：当空缓存重建，别让 notebook 挂掉
        print('向量缓存不可读(%s: %s)，将重新向量化：%s' % (type(e).__name__, e, _CACHE_FILE))
        return {}

def _save_cache(cache):
    """写盘前先与磁盘上已有内容合并，再原子替换 —— 避免多个进程同时跑时互相覆盖 / 写坏文件"""
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    for k, v in _load_cache().items():
        cache.setdefault(k, v)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    # 进程号唯一，别抢同一个临时文件；注意 np.savez_compressed 会自动补 .npz 后缀，临时名必须也是 .npz 结尾
    tmp = _CACHE_FILE.with_name('%s.%d.tmp.npz' % (_CACHE_FILE.stem, os.getpid()))
    np.savez_compressed(tmp, hashes=hs, vectors=vs)
    try:
        os.replace(tmp, _CACHE_FILE)            # 原子替换：别的进程读到的永远是完整文件
    except OSError:                             # 目标被占用时稍等再试
        time.sleep(0.2); os.replace(tmp, _CACHE_FILE)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


语料就绪：7 篇文档 → 57 个片段，向量维度 1024


## 1. 三个最核心的生成指标

| 指标 | 英文 | 问的是 |
|------|------|--------|
| **忠实度** | Faithfulness | 回答是否**忠于检索到的上下文**、没瞎编 |
| **答案相关性** | Answer Relevance | 回答是否**答在点上**（不是废话也相关） |
| **上下文相关性** | Context Relevance | 检索到的上下文**是否用上了/有没有噪声** |

> Faithfulness 是防幻觉的关键闸门：回答里的每个论断都应能溯源到某个检索片段。

## 先用大白话讲一遍（三个指标到底在算什么）

先看一次具体的问答，后面所有算法都是围着它转：

- **问题**：星云支持私有化部署吗？
- **检索到的上下文**：`星云支持公有云与私有化两种部署方式。`
- **模型回答**：`星云既支持公有云也支持私有化部署，而且性能是友商两倍。`

**1）Faithfulness（忠实度）= 回答里的话，有多少能在检索到的上下文里找到依据**

做法：把回答切成一条条「可单独核对的小断言」，逐条问一句"上下文里说了吗？"

| 断言 | 上下文里说了吗 |
|------|----------------|
| 星云支持公有云部署 | 说了 ✅ |
| 星云支持私有化部署 | 说了 ✅ |
| 性能是友商两倍 | 从没说过 ❌ ← **这就是幻觉** |

算：**有依据的 2 条 ÷ 一共 3 条 = 0.67**。注意分母是**断言条数**，不是句子数 —— 切得越细，越能指出到底是哪一句话在瞎编。

**2）Answer Relevance（答案相关性）= 光看这个回答，能不能反推出原问题**

做法：不比对上下文，而是把**回答**交给模型问一句"这段话像是在回答什么问题？"，让它生成几个问题，再和原问题比语义相似度（余弦相似度，越接近 1 越像），取平均。

- 答得越切题 → 反推出的问题越接近原问题 → 分越高；
- 答得越空泛、越跑题 → 反推的问题越东一榔头西一棒 → 分越低。

要注意：它只看「回答 ↔ 问题」这一对，**完全不看上下文**。所以"编得很切题"也能拿高分 —— 这正是它必须和 Faithfulness 配对使用的原因。

**3）Context Relevance（上下文相关性）= 检索来的资料里，有多少是真的有用的**

做法：把上下文切成句子，逐句判断"跟这个问题有关吗"，算相关句子占的比例。

再进一步，**ContextPrecision@K 还看顺序**：同样是 1 条有用，排在第 1 位比排在第 3 位得分更高 —— 因为它算的是"走到每一句时的命中率"再按是否相关加权平均。

**4）三个分数怎么合成一个总分**

最常用的是「RAG 三件套」（RAG Triad）：三个维度各自的平均分再取平均；也可以取**最小值**当红线 —— 任何一维不及格就判整体不通过，适合挂到线上做告警。

> 下面代码块会把这三个指标的中间数字都打印出来（联网时答案相关性用真实 embedding 算），对照着看一遍，再看下一节的公式就顺了。

## 精确计算公式

> **约定**：$Q$ 为评测问题集；$q$ 一个问题；$A$ 为模型回答；$C=\{c_1,\dots,c_m\}$ 为检索到的上下文片段；裁判模型记作 $\mathcal{J}$；所有指标归一到 $[0,1]$，**整集得分 = 各问题得分的算术平均** $S=\frac{1}{|Q|}\sum_{q\in Q}s(q)$。

**1) Faithfulness（忠实度）：回答有没有瞎编**
把回答拆成**原子断言** $\{a_1,\dots,a_n\}$（每条只含一个事实），逐条判断能否由上下文 $C$ 推出：
$$\mathrm{supp}(a)=\mathbb{1}\bigl[\,a\ \text{可由}\ C\ \text{推出}\,\bigr]$$
$$\mathrm{Faithfulness}=\frac{1}{n}\sum_{i=1}^{n}\mathrm{supp}(a_i)=\frac{\bigl|\{a_i:\mathrm{supp}(a_i)=1\}\bigr|}{n}\in[0,1]$$
分母是**断言条数**而非句子数；拆得越细，越能定位幻觉出在哪条论断（本课代码即此式，断言由裁判模型现场从真实回答里拆出来）。

**2) Answer Relevance（答案相关性）：回答有没有答在点上**
RAGAS 的“反向生成”定义——由回答 $A$ 反向生成 $n$ 个候选问题 $\{q_1,\dots,q_n\}$（“这个回答像是在回答什么问题”），与原问题做语义相似度取均值：
$$\mathrm{AnswerRelevance}(q)=\frac{1}{n}\sum_{i=1}^{n}\cos\bigl(\mathbf{e}(q),\ \mathbf{e}(q_i)\bigr)$$
其中 $\mathbf{e}(\cdot)$ 为 embedding 向量，$\cos$ 为余弦相似度。回答越空泛，反推出的问题越分散，均值越低。

**3) Context Relevance（上下文相关性）：检索来的上下文有没有用**
把上下文拆成句子 $\{s_1,\dots,s_t\}$，逐句判断与问题是否相关：
$$\mathrm{ContextRelevance}=\frac{\bigl|\{s_j:\ \mathrm{relevant}(s_j,q)=1\}\bigr|}{t}\in[0,1]$$
**排序敏感版**（RAGAS 的 context precision@K，奖励相关片段排在前面）：
$$\mathrm{ContextPrecision@K}=\frac{\sum_{k=1}^{K}\bigl(\mathrm{Precision@}k\times v_k\bigr)}{\sum_{k=1}^{K}v_k},\qquad v_k=\mathbb{1}\bigl[\text{第 }k\text{ 个片段相关}\bigr]$$
$$\text{其中}\quad \mathrm{Precision@}k=\frac{1}{k}\sum_{j=1}^{k}v_j$$

**4) LLM-as-a-Judge 的打分与聚合**
裁判按评分卡对三维分别给分 $s\in[0,1]$（若给 1~5 分则线性归一 $\frac{s-1}{4}$）：
$$s_{\text{faithful}},\,s_{\text{answer\_rel}},\,s_{\text{context\_rel}}=\mathcal{J}(q,\,C,\,A)$$
整集得分 $\bar{s}=\frac{1}{|Q|}\sum_{q}s(q)$。RAG 三件套（RAG Triad）的综合分可取
$$S_{\text{triad}}=\frac{1}{3}\Bigl(\overline{s_{\text{faithful}}}+\overline{s_{\text{answer\_rel}}}+\overline{s_{\text{context\_rel}}}\Bigr)$$
或按业务取三者**最小值**当红线（任一维不达标即判不通过）：
$$S_{\min}=\min\bigl(\overline{s_{\text{faithful}}},\ \overline{s_{\text{answer\_rel}}},\ \overline{s_{\text{context\_rel}}}\bigr)$$

> 评测要固定四件事才可比：**裁判模型、评分卡 prompt、温度（建议 0~0.1）、输出解析方式**。裁判本身有位置偏好与长度偏好，需固定资料顺序、统一答案格式。

In [3]:
# 手算演示：在真实语料上跑一遍三个指标，把中间量逐个打印出来
# 与上一节公式一一对应：judge_claims→(1)、answer_relevance→(2)、judge_sentences→(3)、context_precision→(3) 排序敏感版
# 评测集与 33 课同一份（data/评测集.md，人工标注）；这里评估的是「生成」，不是「检索」
EVAL_Q = [m.group(1) for line in Path('data/评测集.md').read_text(encoding='utf-8').splitlines()
          if (m := re.match(r'^\|\s*\d+\s*\|\s*(.+?)\s*\|', line))]


def retrieve_context(q, top_n=3):
    """真实检索：混合召回 Top-10 再用 qwen3-rerank 精排 Top-N（生成评估的输入必须是真的检索结果）"""
    return rerank(q, [c['text'] for c in hybrid_retrieve(q, 10)], top_n=top_n)


def gen_answer(q, ctx):
    """真调 qwen-plus 生成回答：只依据检索到的上下文"""
    return chat('【参考资料】\n%s\n\n【问题】%s' % ('\n'.join(ctx), q))


def judge_claims(answer, ctx):
    """(1) 把回答拆成原子断言，逐条判断能否由上下文推出 —— 这就是 RAGAS 算 Faithfulness 的真实机制"""
    out = chat_json(
        '把下面这段"模型回答"拆成若干条可独立核验的事实断言，再对照"检索到的上下文"，'
        '逐条标记 supported=true/false（上下文没有依据就标 false）。\n'
        '上下文: %s\n模型回答: %s\n'
        '只输出 JSON：{"claims": [{"claim": "<断言>", "supported": true 或 false}]}'
        % ('\n'.join(ctx), answer),
        system='你是 RAG 忠实度(Faithfulness)评估器。判定只依据给定上下文，上下文没提到的内容一律标 false，'
               '不自行补充知识。只输出 JSON，禁止输出其它文字。', temperature=0.1)
    return (out or {}).get('claims', [])


def judge_reverse_questions(answer, n=3):
    """(2) 由回答反推"它像是在回答什么问题"（RAGAS 的 answer relevance 做法）"""
    out = chat_json('下面这段回答像是在回答什么问题？给出 %d 个不同问法的问句。\n回答: %s\n'
                    '只输出 JSON：{"questions": ["...", "..."]}' % (n, answer),
                    system='你只输出 JSON，不要解释。', temperature=0.1)
    return (out or {}).get('questions', [])


def cosine(a, b):
    """真调 text-embedding-v3 算余弦：底座 embed 已 L2 归一化，点积即余弦"""
    v = embed([a, b])
    return float(v[0] @ v[1])


def judge_sentences(q, ctx):
    """(3) 把上下文拆成句子，逐句判断与问题是否相关"""
    raw = [s.strip() for s in re.split(r'(?<=[。！？\n])', '\n'.join(ctx))]
    # 只保留真句子：滤掉 `**`、`## ` 这类纯 Markdown 符号行，否则会拉低分母、让分数失真
    sents = [s for s in raw if re.search(r'[\w\u4e00-\u9fff]', s)]
    out = chat_json('判断下面每个句子是否与问题相关，逐句给出 true/false。\n问题: %s\n句子: %s\n'
                    '只输出 JSON：{"relevant": [true, false, ...]}' % (q, sents),
                    system='你只输出 JSON，不要解释。', temperature=0.1)
    return sents, [bool(x) for x in (out or {}).get('relevant', [])]


def judge_chunks(q, ctx):
    """排序敏感版用：判断每个检索片段是否相关（列表顺序就是检索顺序）"""
    out = chat_json('逐条判断下面每个检索片段是否与问题相关。\n问题: %s\n片段: %s\n'
                    '只输出 JSON：{"relevant": [true, false, ...]}' % (q, ctx),
                    system='你只输出 JSON，不要解释。', temperature=0.1)
    return [bool(x) for x in (out or {}).get('relevant', [])]


def context_precision(v):
    """ContextPrecision@K = Σ(Precision@k × v_k) / Σ v_k —— 相关片段排得越前，分越高"""
    if not v or not sum(v): return 0.0
    return sum((sum(v[:k]) / k) * v[k - 1] for k in range(1, len(v) + 1)) / sum(v)


Q0 = EVAL_Q[0]
if not _HAS_KEY:
    recorded("""【问题】 星云客服机器人能私有化部署吗
【检索到的上下文（真检索 + qwen3-rerank 精排 Top-3）】（节选）
  [1] ## 部署方式 产品支持公有云 SaaS 与私有化两种部署方式。…
  [2] ## 部署相关 **Q：能私有化部署吗？** A：能。标准版及以上可申请私有化部署…
  [3] ## 数据与安全 - 公有云客户数据在传输与存储两侧均加密…
【模型回答】 能。星云客服机器人支持私有化部署，标准版及以上可申请私有化部署，需联系销售单独开通；
  企业版则标配私有化部署能力，并包含一次现场实施。私有化版本部署在客户自有服务器或专有云上…
【1) Faithfulness】真调裁判拆断言并逐条核验：
  「星云客服机器人支持私有化部署」   上下文里有依据 ✅
  「标准版及以上可申请私有化部署」   上下文里有依据 ✅
  「私有化部署需联系销售单独开通」   上下文里有依据 ✅
  「企业版标配私有化部署能力」       上下文里有依据 ✅
  「企业版包含一次现场实施」         上下文里有依据 ✅
  「私有化版本部署在客户自有服务器或专有云上」 上下文里有依据 ✅
  「私有化版本数据全程不出内网」     上下文里有依据 ✅
  「私有化版本的模型可对接客户自建的推理服务」 上下文里有依据 ✅
  算式：有依据 8 ÷ 断言总数 8 = 1.00   ← 分母是断言条数，拆得越细越能定位幻觉
【2) Answer Relevance】真调反推问题，再用 text-embedding-v3 算真实余弦：
  反推问题「星云客服机器人是否支持私有化部署？」 相似度 = 0.985
  反推问题「不同版本的星云客服机器人在私有化部署方面有哪些差异？」 相似度 = 0.888
  反推问题「私有化部署后，数据安全性和模型部署方式是怎样的？」 相似度 = 0.684
  算式：(0.985 + 0.888 + 0.684) ÷ 3 = 0.852   ← 反推得越偏，均分越低
【3) Context Relevance】逐句判定（节选，共 20 句）：
  句子「## 部署方式」→ 相关 ✅
  句子「产品支持公有云 SaaS 与私有化两种部署方式。」→ 相关 ✅
  句子「- **公有云 SaaS**：默认方式，共享集群，开箱即用；」→ 跑题 ❌
  句子「- **专有云**：独立集群部署在星云托管的云环境中，网络与存储隔离；」→ 跑题 ❌
  句子「- 支持敏感信息自动脱敏（手机号、身份证号、银行卡号）；」→ 跑题 ❌
  算式：相关句数 12 ÷ 句子总数 20 = 0.60
【3b) ContextPrecision@3】还看顺序：
  片段判定 v = [1, 1, 0]
  第 1 个片段 相关=1 → Precision@1 = 1/1 = 1.00
  第 2 个片段 相关=1 → Precision@2 = 2/2 = 1.00
  第 3 个片段 相关=0 → Precision@3 = 2/3 = 0.67（v=0，不计入分子）
  算式：ContextPrecision@3 = 2.00 ÷ 相关片段数 2 = 1.00
  对照：同样 2 条相关片段若排在第 1、2 位 → ContextPrecision@3 = 1.00
  对照：同样 2 条相关片段若排在第 1、3 位 → ContextPrecision@3 = 0.83""",
             '录制于 2026-09-12，模型 qwen-plus + text-embedding-v3')
else:
    print('【问题】', Q0)
    ctx = [t for t, _s in retrieve_context(Q0)]
    ans = gen_answer(Q0, ctx)
    print('\n【检索到的上下文（真检索 + qwen3-rerank 精排 Top-3）】')
    for i, c in enumerate(ctx, 1):
        print('  [%d] %s' % (i, c[:60].replace('\n', ' ') + '…'))
    print('\n【模型回答】', ans)

    # ---------- 1) Faithfulness ----------
    print('\n【1) Faithfulness】真调裁判拆断言并逐条核验：')
    claims = judge_claims(ans, ctx)
    sup = 0
    for c in claims:
        ok = bool(c.get('supported'))
        sup += ok
        print('  「%s」   %s' % (c.get('claim'), '上下文里有依据 ✅' if ok else '上下文里没有依据 → 判为幻觉 ❌'))
    print('  算式：有依据 %d ÷ 断言总数 %d = %.2f   ← 分母是断言条数，拆得越细越能定位幻觉'
          % (sup, len(claims), sup / max(len(claims), 1)))

    # ---------- 2) Answer Relevance ----------
    print('\n【2) Answer Relevance】真调反推问题，再用 text-embedding-v3 算真实余弦：')
    gqs = judge_reverse_questions(ans)
    sims = [cosine(Q0, g) for g in gqs]
    for g, s in zip(gqs, sims):
        print('  反推问题「%s」 相似度 = %.3f' % (g, s))
    print('  算式：(%s) ÷ %d = %.3f   ← 反推得越偏，均分越低'
          % (' + '.join('%.3f' % s for s in sims), len(sims), sum(sims) / max(len(sims), 1)))

    # ---------- 3) Context Relevance + ContextPrecision@K ----------
    print('\n【3) Context Relevance】逐句判定：')
    sents, rel = judge_sentences(Q0, ctx)
    for s, r in zip(sents, rel):
        print('  句子「%s」→ %s' % (s[:52], '相关 ✅' if r else '跑题 ❌'))
    print('  算式：相关句数 %d ÷ 句子总数 %d = %.2f' % (sum(rel), len(rel), sum(rel) / max(len(rel), 1)))

    print('\n【3b) ContextPrecision@3】还看顺序：')
    v = judge_chunks(Q0, ctx)
    print('  片段判定 v =', [int(x) for x in v])
    num = 0.0
    for k_, vk in enumerate(v, 1):
        p_at_k = sum(v[:k_]) / k_
        num += p_at_k * vk
        print('  第 %d 个片段 相关=%d → Precision@%d = %d/%d = %.2f%s'
              % (k_, vk, k_, sum(v[:k_]), k_, p_at_k, '（v=0，不计入分子）' if not vk else ''))
    print('  算式：ContextPrecision@3 = %.2f ÷ 相关片段数 %d = %.2f'
          % (num, sum(v), context_precision(v)))
    for name, vb in (('第 1、2 位', [1, 1, 0]), ('第 1、3 位', [1, 0, 1])):
        print('  对照：同样 %d 条相关片段若排在%s → ContextPrecision@3 = %.2f'
              % (sum(vb), name, context_precision(vb)))

【问题】 星云客服机器人能私有化部署吗

【检索到的上下文（真检索 + qwen3-rerank 精排 Top-3）】
  [1] ## 部署方式 产品支持公有云 SaaS 与私有化两种部署方式。公有云版本开通即可使用，按调用量计费；私有化版本部署在客…
  [2] ## 部署相关 **Q：能私有化部署吗？** A：能。标准版及以上可申请私有化部署，需联系销售单独开通；企业版标配私有化…
  [3] ## 数据与安全 - 公有云客户数据在传输与存储两侧均加密，不同租户之间逻辑隔离； - 私有化部署数据全程不出客户内网，…

【模型回答】 能。星云客服机器人支持私有化部署，标准版及以上可申请私有化部署，需联系销售单独开通；企业版则标配私有化部署能力，并包含一次现场实施。私有化版本部署在客户自有服务器或专有云上，数据全程不出内网，模型亦可对接客户自建的推理服务。

【1) Faithfulness】真调裁判拆断言并逐条核验：
  「星云客服机器人支持私有化部署」   上下文里有依据 ✅
  「标准版及以上可申请私有化部署」   上下文里有依据 ✅
  「私有化部署需联系销售单独开通」   上下文里有依据 ✅
  「企业版标配私有化部署能力」   上下文里有依据 ✅
  「企业版包含一次现场实施」   上下文里有依据 ✅
  「私有化版本部署在客户自有服务器或专有云上」   上下文里有依据 ✅
  「私有化版本数据全程不出内网」   上下文里有依据 ✅
  「私有化版本的模型可对接客户自建的推理服务」   上下文里有依据 ✅
  算式：有依据 8 ÷ 断言总数 8 = 1.00   ← 分母是断言条数，拆得越细越能定位幻觉

【2) Answer Relevance】真调反推问题，再用 text-embedding-v3 算真实余弦：
  反推问题「星云客服机器人是否支持私有化部署？」 相似度 = 0.985
  反推问题「不同版本的星云客服机器人在私有化部署方面有哪些差异？」 相似度 = 0.888
  反推问题「私有化部署后，数据安全性和模型部署方式是怎样的？」 相似度 = 0.684
  算式：(0.985 + 0.888 + 0.684) ÷ 3 = 0.852   ← 反推得越偏，均分越低

【3) Context Relevance】逐句判定：
  句子「##

In [4]:
# 批量评测：同一套裁判、同一套评分卡，在评测集上跑出一组可比的分数
# 生成类指标天然依赖裁判，所以必须固定「裁判模型 + prompt + 温度」——换一个分数就不可比了
def evaluate_one(q):
    ctx = [t for t, _s in retrieve_context(q)]
    ans = gen_answer(q, ctx)
    claims = judge_claims(ans, ctx)
    faith = sum(1 for c in claims if c.get('supported')) / max(len(claims), 1)
    gqs = judge_reverse_questions(ans)
    ans_rel = float(np.mean([cosine(q, g) for g in gqs])) if gqs else 0.0
    _sents, rel = judge_sentences(q, ctx)
    ctx_rel = sum(rel) / max(len(rel), 1)
    return {'q': q, 'ans': ans, 'faith': faith, 'ans_rel': ans_rel, 'ctx_rel': ctx_rel}


BATCH = EVAL_Q[1:4]        # 换一批问题，避免和上面那条重复调用
if not _HAS_KEY:
    recorded("""问题                           Faithfulness  AnswerRel  ContextRel
标准版多少钱一个月                          1.000      0.925       0.120
知识库容量超了怎么收费                        1.000      0.917       0.875
私有化部署需要什么服务器配置                     0.929      0.735       0.391
------------------------------------------------------------------
整集均值                                0.976      0.859       0.462

RAG Triad 综合分（三维再取平均） = 0.766
RAG Triad 红线分（三维取最小值）  = 0.462   ← 线上做告警时用这个：任何一维不达标即判整体不通过

→ 注意本次运行里 Context Relevance 只有 0.462，而 Faithfulness 是 0.976：回答完全忠于资料、
  也答在点上，但资料本身“夹带”了无关内容。原因是本课的“上下文”= 整段检索片段，一个片段讲一整节，
  里面自然混着与当前问题无关的句子；这正是第 23 课要讲上下文压缩的原因 —— 检索没错，是粒度太粗。
  （表中第 3 行 Faithfulness 0.929 是一次真实幻觉：那条回答里有一句在资料里找不到依据。）""",
             '录制于 2026-09-12，裁判模型 qwen-plus，温度 0.1')
else:
    rows = [evaluate_one(q) for q in BATCH]
    print('%-28s %12s %10s %11s' % ('问题', 'Faithfulness', 'AnswerRel', 'ContextRel'))
    for r in rows:
        print('%-28s %12.3f %10.3f %11.3f' % (r['q'], r['faith'], r['ans_rel'], r['ctx_rel']))
    m = {k: float(np.mean([r[k] for r in rows])) for k in ('faith', 'ans_rel', 'ctx_rel')}
    print('-' * 66)
    print('%-28s %12.3f %10.3f %11.3f' % ('整集均值', m['faith'], m['ans_rel'], m['ctx_rel']))
    triad = float(np.mean(list(m.values())))
    print('\nRAG Triad 综合分（三维再取平均） = %.3f' % triad)
    print('RAG Triad 红线分（三维取最小值）  = %.3f   ← 线上做告警时用这个：任何一维不达标即判整体不通过'
          % min(m.values()))
    print('\n→ 三个维度看的是三件不同的事：忠实度防幻觉、答案相关性防跑题、上下文相关性盯检索质量。'
          '只看平均分会掩盖短板，所以生产上常常两个都报。')
    print('→ 注意本次运行里 Context Relevance 只有 %.3f，而 Faithfulness 是 %.3f：'
          '回答完全忠于资料、也答在点上，但资料本身“夹带”了无关内容。'
          % (m['ctx_rel'], m['faith']))
    print('  原因是本课的“上下文”= 整段检索片段，一个片段讲一整节，里面自然混着与当前问题无关的句子；'
          '这正是第 23 课要讲上下文压缩的原因 —— 检索没错，是粒度太粗。')

问题                           Faithfulness  AnswerRel  ContextRel
标准版多少钱一个月                           1.000      0.925       0.120
知识库容量超了怎么收费                         1.000      0.921       0.875
私有化部署需要什么服务器配置                      1.000      0.829       0.391
------------------------------------------------------------------
整集均值                                1.000      0.892       0.462

RAG Triad 综合分（三维再取平均） = 0.785
RAG Triad 红线分（三维取最小值）  = 0.462   ← 线上做告警时用这个：任何一维不达标即判整体不通过

→ 三个维度看的是三件不同的事：忠实度防幻觉、答案相关性防跑题、上下文相关性盯检索质量。只看平均分会掩盖短板，所以生产上常常两个都报。
→ 注意本次运行里 Context Relevance 只有 0.462，而 Faithfulness 是 1.000：回答完全忠于资料、也答在点上，但资料本身“夹带”了无关内容。
  原因是本课的“上下文”= 整段检索片段，一个片段讲一整节，里面自然混着与当前问题无关的句子；这正是第 23 课要讲上下文压缩的原因 —— 检索没错，是粒度太粗。


In [5]:
# 知识点·真调说明：断言级 Faithfulness —— 幻觉到底出在哪一条论断
# 上面的 judge_claims 用了同一个机制；这里把裁判返回的原始 JSON 摊开看
import json as _json
q = EVAL_Q[0]
if not _HAS_KEY:
    recorded("""裁判模型返回的原始 JSON：
{"claims": [
  {"claim": "星云客服机器人支持私有化部署", "supported": true},
  {"claim": "标准版及以上可申请私有化部署", "supported": true},
  {"claim": "私有化部署需联系销售单独开通", "supported": true},
  {"claim": "企业版标配私有化部署能力", "supported": true},
  {"claim": "企业版包含一次现场实施", "supported": true},
  {"claim": "私有化版本部署在客户自有服务器或专有云上", "supported": true},
  {"claim": "私有化版本数据全程不出内网", "supported": true},
  {"claim": "私有化版本的模型可对接客户自建的推理服务", "supported": true}]}
模型把回答拆成 8 条断言，其中 8 条在上下文里有依据。
Faithfulness = 1.00
被判"无依据"的断言（幻觉）：无""",
             '录制于 2026-09-12，裁判模型 qwen-plus，温度 0.1')
else:
    ctx = [t for t, _s in retrieve_context(q)]
    ans = gen_answer(q, ctx)
    raw = chat('把下面这段"模型回答"拆成若干条可独立核验的事实断言，再对照"检索到的上下文"，'
               '逐条标记 supported=true/false（上下文没有依据就标 false）。\n'
               '上下文: %s\n模型回答: %s\n'
               '只输出 JSON：{"claims": [{"claim": "<断言>", "supported": true 或 false}]}'
               % ('\n'.join(ctx), ans),
               system='你是 RAG 忠实度(Faithfulness)评估器。判定只依据给定上下文，'
                      '上下文没提到的内容一律标 false。只输出 JSON。', temperature=0.1)
    print('【问题】', q)
    print('【模型回答】', ans)
    print('\n裁判模型返回的原始 JSON：')
    print(raw)
    try:
        claims = _json.loads(raw[raw.find('{'): raw.rfind('}') + 1])['claims']
        sup = sum(1 for c in claims if c['supported'])
        print('\n模型把回答拆成 %d 条断言，其中 %d 条在上下文里有依据。' % (len(claims), sup))
        print('Faithfulness = %.2f' % (sup / len(claims)))
        unsup = [c['claim'] for c in claims if not c['supported']]
        print('被判"无依据"的断言（幻觉）：', '；'.join(unsup) if unsup else '无')
    except Exception as e:
        print('未解析成 JSON：', e, '—— 说明需在 prompt 里收紧输出格式。')
print('→ 先拆断言、再逐条核验，幻觉出在哪个论断一目了然 —— 这就是 RAGAS / DeepEval 算 Faithfulness 的真实机制。')

【问题】 星云客服机器人能私有化部署吗
【模型回答】 能。星云客服机器人支持私有化部署，标准版及以上可申请私有化部署，需联系销售单独开通；企业版则标配私有化部署能力，并包含一次现场实施。私有化版本部署在客户自有服务器或专有云上，数据全程不出内网，模型可对接客户自建的推理服务。

裁判模型返回的原始 JSON：
{
  "claims": [
    {
      "claim": "星云客服机器人支持私有化部署",
      "supported": true
    },
    {
      "claim": "标准版及以上可申请私有化部署",
      "supported": true
    },
    {
      "claim": "私有化部署需联系销售单独开通",
      "supported": true
    },
    {
      "claim": "企业版标配私有化部署能力",
      "supported": true
    },
    {
      "claim": "企业版包含一次现场实施",
      "supported": true
    },
    {
      "claim": "私有化版本部署在客户自有服务器或专有云上",
      "supported": true
    },
    {
      "claim": "私有化版本数据全程不出内网",
      "supported": true
    },
    {
      "claim": "私有化版本的模型可对接客户自建的推理服务",
      "supported": true
    }
  ]
}

模型把回答拆成 8 条断言，其中 8 条在上下文里有依据。
Faithfulness = 1.00
被判"无依据"的断言（幻觉）： 无
→ 先拆断言、再逐条核验，幻觉出在哪个论断一目了然 —— 这就是 RAGAS / DeepEval 算 Faithfulness 的真实机制。


In [6]:
# LLM-as-a-Judge：换一种判法 —— 不让模型拆断言，直接按评分卡给 0~1 分
# 对比上面逐条核验的结果：同一份问答，两种判法分数往往不同，所以「固定判法」比「判法本身」更重要
def judge(question, context, answer):
    """让裁判模型按三维评分卡打分，返回解析后的 JSON（温度压到 0.1，保证可复现）"""
    return chat_json(
        '你是 RAG 评估裁判。\n问题: %s\n上下文: %s\n回答: %s\n'
        '分别对 faithfulness(忠于上下文程度)、answer_relevance(切题程度)、'
        'context_relevance(上下文相关程度) 打分 0-1，只输出 JSON 如 '
        '{"faithfulness":0.9,"answer_relevance":0.9,"context_relevance":0.9}' % (question, context, answer),
        system='你是严格的 RAG 评估裁判，只输出 JSON，不要解释。', temperature=0.1)

q = EVAL_Q[0]
if not _HAS_KEY:
    recorded("""裁判评分卡给出的分数：{'faithfulness': 1.0, 'answer_relevance': 1.0, 'context_relevance': 1.0}
RAG Triad 均值 = 1.000
同一份上下文，换「逐句核验」判法算 Context Relevance = 12/20 = 0.60

→ context_relevance 从 1.00 变成 0.60：整体打分给的是“大概多好”的粗略观感，
  逐句核验给的是“哪几句跑题”的可定位证据。""",
             '录制于 2026-09-12，裁判模型 qwen-plus，温度 0.1')
else:
    ctx = [t for t, _s in retrieve_context(q)]
    ans = gen_answer(q, ctx)
    card = judge(q, '\n'.join(ctx), ans)
    print('裁判评分卡给出的分数：', card)
    if card:
        vals = [float(v) for v in card.values()]
        print('RAG Triad 均值 = %.3f' % (sum(vals) / len(vals)))
    _s, rel = judge_sentences(q, ctx)          # 同一个问题，换成逐句核验的判法
    fine = sum(rel) / max(len(rel), 1)
    print('同一份上下文，换「逐句核验」判法算 Context Relevance = %d/%d = %.2f' % (sum(rel), len(rel), fine))
    if card:
        print('\n→ 同一份问答、同一个裁判模型，只是换了判法（整体打分 vs 逐句核验），'
              'context_relevance 从 %.2f 变成 %.2f：' % (float(card.get('context_relevance', 0)), fine))
    print('  整体打分给的是"大概多好"的粗略观感，逐句核验给的是"哪几句跑题"的可定位证据；')
    print('  所以评测报告里必须写清用了哪种判法 —— 这正是"固定裁判模型、评分卡 prompt、温度、解析方式"的由来。')

裁判评分卡给出的分数： {'faithfulness': 1.0, 'answer_relevance': 1.0, 'context_relevance': 1.0}
RAG Triad 均值 = 1.000
同一份上下文，换「逐句核验」判法算 Context Relevance = 12/20 = 0.60

→ 同一份问答、同一个裁判模型，只是换了判法（整体打分 vs 逐句核验），context_relevance 从 1.00 变成 0.60：
  整体打分给的是"大概多好"的粗略观感，逐句核验给的是"哪几句跑题"的可定位证据；
  所以评测报告里必须写清用了哪种判法 —— 这正是"固定裁判模型、评分卡 prompt、温度、解析方式"的由来。


## 2. 评估框架

| 框架 | 特色 | 定位 |
|------|------|------|
| **RAGAS** | 指标体系最贴 RAG（faithfulness/context precision…） | 离线指标库 |
| **DeepEval** | pytest 风格、断言式 | 单测化评估 |
| **TruLens** | 反馈函数 + 可视化追踪 | 反馈/追踪 |
| **LangSmith** | 线上 trace + 标注 + 数据集回归 | 生产观测 |
| **RAGChecker** | 细粒度诊断（噪声敏感度等） | 深度诊断 |

> 同一套**评测集**固定后，框架只是帮你把“指标算出来”。

## 3. 开源数据集与基准

| 数据集 | 内容 | 用途 |
|--------|------|------|
| **MS MARCO** | 必应搜索真实查询+段落 | 检索/排序基准 |
| **BEIR** | 18 个异构任务合集 | 零样本泛化能力 |
| **Natural Questions** | Google 搜索问答 | 开放域问答 |
| **KILT / TriviaQA** | 知识密集任务 | 知识型 RAG |

## 小结

- 生成三指标：**Faithfulness / Answer Relevance / Context Relevance**；
- 主流做法是 **LLM-as-a-Judge**，打分卡要固定、可复现；
- 框架（RAGAS 等）与基准（MS MARCO/BEIR）解决“怎么算”“在哪比”。